# Week 6, Day 3: Diagonalisation and matrix powers: what repeated connectivity does

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w006/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
A = np.array([[0.7, 0.3],
              [0.2, 0.6]])
lam, V = np.linalg.eig(A)
print("eigenvalues:", lam.round(3))
print("V Λ V⁻¹ =\n", (V @ np.diag(lam) @ np.linalg.inv(V)).round(3))

### Exercise: Matrix powers through eigenvalues

Write `eig_power(A, k)` that computes $A^k$ as $V\Lambda^kV^{-1}$ using `np.linalg.eig`, without calling `matrix_power`. Return the real part (tiny imaginary rounding errors can appear).

In [ ]:
def eig_power(A, k):
    lam, V = np.linalg.eig(A)
    ...

print(eig_power(A, 10).round(4))
print(np.linalg.matrix_power(A, 10).round(4))

In [ ]:
import numpy as np
import dis
def test_match():
    "matches np.linalg.matrix_power for several k"
    M = np.array([[0.7, 0.3], [0.2, 0.6]])
    for k in [0, 1, 5, 40]:
        assert np.allclose(eig_power(M, k), np.linalg.matrix_power(M, k))

def test_3d():
    "works for a 3 by 3 matrix"
    M = np.array([[0.5, 0.3, 0.0], [0.3, 0.4, 0.2], [0.0, 0.2, 0.6]])
    assert np.allclose(eig_power(M, 12), np.linalg.matrix_power(M, 12))

def test_no_matrix_power():
    "does not call matrix_power"
    names = {i.argval for i in dis.get_instructions(eig_power)}
    assert "matrix_power" not in names

test_match()
test_3d()
test_no_matrix_power()
print("All checks passed")

### Exercise: Simulate through modes

Starting from `x0`, compute the eigen-coordinates `c0`, then build `traj` (31 by 2) whose row `t` is $\sum_i c_{0,i}\lambda_i^t\mathbf{v}_i$, using no loop over matrix multiplications (a loop over t is fine, or use broadcasting). Finally compute each mode's half-life in steps.

In [ ]:
import matplotlib.pyplot as plt
x0 = np.array([1.0, -0.5])
c0 = ...
ts = np.arange(31)
traj = ...                      # shape (31, 2)
half_lives = ...

plt.figure(figsize=(5, 3))
plt.plot(ts, traj[:, 0], label="population 1"); plt.plot(ts, traj[:, 1], label="population 2")
plt.xlabel("step"); plt.legend(frameon=False); plt.show()
print("eigenvalues", lam.round(3), " half-lives", np.round(half_lives, 2))

In [ ]:
import numpy as np
def test_coords():
    "c0 holds the eigen-coordinates of x0"
    assert np.allclose(V @ c0, x0)

def test_traj():
    "traj from the modes matches direct simulation"
    x = x0.copy(); direct = [x]
    for t in range(30):
        x = A @ x; direct.append(x)
    assert traj.shape == (31, 2) and np.allclose(traj, np.array(direct))

def test_half():
    "half_lives matches ln(1/2) / ln(lambda) for each mode"
    assert np.allclose(np.sort(half_lives), np.sort(np.log(0.5) / np.log(lam)))

test_coords()
test_traj()
test_half()
print("All checks passed")

### Exercise: Transient amplification

Kick each circuit with the unit vector `x0 = (0, 1)` and record the length of the activity $\lVert B^t\mathbf{x}_0 \rVert$ for t = 0 to 40 in `norms_B`. Do the same for `S`, a diagonal matrix with the same eigenvalues, in `norms_S`. Plot both.

In [ ]:
B = np.array([[0.9, 2.0], [0.0, 0.8]])
S = np.diag([0.9, 0.8])
x0 = np.array([0.0, 1.0])
norms_B, norms_S = [], []
xb, xs = x0.copy(), x0.copy()
for t in range(41):
    ...

plt.figure(figsize=(5, 3))
plt.plot(norms_B, label="non-normal B"); plt.plot(norms_S, label="symmetric, same eigenvalues")
plt.xlabel("step"); plt.ylabel("|x_t|"); plt.legend(frameon=False); plt.show()
print("peak", round(max(norms_B), 2), "at step", int(np.argmax(norms_B)))

In [ ]:
import numpy as np
def test_norms():
    "norms_B and norms_S have 41 values each, starting at 1"
    assert len(norms_B) == 41 and len(norms_S) == 41
    assert np.isclose(norms_B[0], 1) and np.isclose(norms_S[0], 1)

def test_amplify():
    "the non-normal circuit grows to more than 3 times the kick before decaying"
    assert max(norms_B) > 3 and norms_B[-1] < max(norms_B)

def test_symmetric():
    "the symmetric circuit with the same eigenvalues never grows"
    assert np.all(np.diff(norms_S) <= 1e-12)

test_norms()
test_amplify()
test_symmetric()
print("All checks passed")